# KPI de Calificaciones (punto 6.2)

Calcula los KPI de calificaciones a partir de `data/integrado/`. Las funciones son puras (reciben y devuelven DataFrames); la lectura de archivos se hace solo en la celda de carga.

KPI de este notebook:
- Promedio general por asignatura
- Distribución de notas (histograma)
- Porcentaje de estudiantes promocionados, regulares y libres
- Comparación de resultados entre asignaturas

## 1. Carga de datos

In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path("../../data")
df_calificaciones = pd.read_parquet(DATA / "integrado" / "calificaciones_integrado.parquet")
df_cruce = pd.read_parquet(DATA / "integrado" / "cruce_matriculados_integrado.parquet")

## 2. Promedio general por asignatura

Las asignaturas no usan la misma escala ni todos los instrumentos valen lo mismo: en Álgebra las autoevaluaciones valen 10 y el parcial 25, mientras que en Probabilidad y Algoritmos cada instrumento vale 100. Para respetar ese peso relativo, la nota de cada estudiante se calcula sobre el total de puntos, no como promedio de porcentajes:

`nota_pct = suma(nota_obtenida) / suma(nota_max) * 100`

De esta forma el resultado queda en una escala de 0 a 100 y un instrumento que vale más (por ejemplo un parcial de 25 frente a una autoevaluación de 10) pesa proporcionalmente más. El promedio de la asignatura es el promedio de las notas de los estudiantes, de modo que cada estudiante pesa igual. Por defecto se consideran solo los estudiantes matriculados.

In [ ]:
def solo_matriculados(calificaciones: pd.DataFrame, cruce: pd.DataFrame) -> pd.DataFrame:
    """Deja únicamente las calificaciones de estudiantes matriculados en esa asignatura."""
    matriculados = cruce.loc[cruce["matriculado"], ["materia", "usuario"]]
    return calificaciones.merge(matriculados, on=["materia", "usuario"], how="inner")


def nota_por_estudiante(calificaciones: pd.DataFrame) -> pd.DataFrame:
    """Nota (0-100) de cada estudiante en cada asignatura sobre el total de puntos posibles.

    nota_pct = suma(nota_obtenida) / suma(nota_max) * 100, considerando solo los
    instrumentos que el estudiante rindió. Las filas sin nota o sin máximo válido
    (nulo o <= 0) se ignoran.

    Devuelve: materia, usuario, instrumentos, puntos_obtenidos, puntos_posibles, nota_pct.
    """
    df = calificaciones[calificaciones["nota_max"] > 0].dropna(subset=["nota_obtenida", "nota_max"])
    res = (
        df.groupby(["materia", "usuario"])
        .agg(instrumentos=("nota_max", "size"), puntos_obtenidos=("nota_obtenida", "sum"), puntos_posibles=("nota_max", "sum"))
        .reset_index()
    )
    res["nota_pct"] = res["puntos_obtenidos"] / res["puntos_posibles"] * 100
    return res


def promedio_general_por_asignatura(calificaciones: pd.DataFrame, cruce: pd.DataFrame,
                                    matriculados: bool = True) -> pd.DataFrame:
    """KPI: promedio general por asignatura, en porcentaje del máximo posible (0-100).

    Devuelve una fila por asignatura: estudiantes evaluados, promedio_pct (promedio de
    la nota de cada estudiante) y mediana_pct.
    """
    if matriculados:
        calificaciones = solo_matriculados(calificaciones, cruce)
    por_estudiante = nota_por_estudiante(calificaciones)
    return (
        por_estudiante.groupby("materia")["nota_pct"]
        .agg(estudiantes="size", promedio_pct="mean", mediana_pct="median")
        .round(2)
        .reset_index()
    )

In [ ]:
promedio_asignatura = promedio_general_por_asignatura(df_calificaciones, df_cruce)
promedio_asignatura

### Detalle por tipo de evaluación

Apertura del mismo promedio según el tipo de evaluación, con el mismo criterio (suma de puntos obtenidos sobre suma de puntos posibles, por estudiante, y luego promedio entre estudiantes), para ver qué instrumentos explican el resultado de cada asignatura.

In [ ]:
def promedio_por_tipo_evaluacion(calificaciones: pd.DataFrame, cruce: pd.DataFrame,
                                 matriculados: bool = True) -> pd.DataFrame:
    """Promedio (0-100) por asignatura y tipo de evaluación, con el máximo original del instrumento."""
    if matriculados:
        calificaciones = solo_matriculados(calificaciones, cruce)
    df = calificaciones[calificaciones["nota_max"] > 0].dropna(subset=["nota_obtenida", "nota_max"])
    por_estudiante = (
        df.groupby(["materia", "tipo_evaluacion", "usuario"])
        .agg(obtenidos=("nota_obtenida", "sum"), posibles=("nota_max", "sum"), nota_max=("nota_max", "max"))
        .reset_index()
    )
    por_estudiante["nota_pct"] = por_estudiante["obtenidos"] / por_estudiante["posibles"] * 100
    return (
        por_estudiante.groupby(["materia", "tipo_evaluacion"])
        .agg(estudiantes=("usuario", "size"), nota_max=("nota_max", "max"), promedio_pct=("nota_pct", "mean"))
        .round(2)
        .reset_index()
    )


promedio_por_tipo_evaluacion(df_calificaciones, df_cruce)

### Control

Verifica que el porcentaje por calificación (`nota_obtenida / nota_max * 100`) coincida con la columna `porcentaje` ya presente en los datos y que no haya valores fuera de 0-100 ni notas sin máximo válido.

In [ ]:
pct = df_calificaciones["nota_obtenida"] / df_calificaciones["nota_max"] * 100
print("Máx. diferencia vs. columna 'porcentaje':", (pct - df_calificaciones["porcentaje"]).abs().max().round(4))
print("Fuera de rango 0-100:", int((~pct.between(0, 100)).sum()))
print("Sin nota o sin nota_max válido:", int((pct.isna() | (df_calificaciones["nota_max"] <= 0)).sum()))

## 3. Distribución de notas (histograma)

Distribuye la nota de cada estudiante (`nota_pct`, 0-100, calculada como en el KPI anterior) en rangos de `ANCHO_BIN` puntos. Cada estudiante cuenta una vez por asignatura. Los rangos son `[0, 10)`, `[10, 20)`, ..., `[90, 100]`; el último incluye el 100.

La función devuelve una tabla en formato largo (una fila por asignatura y rango) con todos los rangos presentes aunque estén vacíos. Es el insumo directo del histograma de Streamlit: `histogram(tabla[tabla["materia"] == materia], "rango", "estudiantes", ...)` de `ui/components/charts.py`.

In [ ]:
def distribucion_de_notas(calificaciones: pd.DataFrame, cruce: pd.DataFrame,
                          ancho_bin: int = 10, matriculados: bool = True) -> pd.DataFrame:
    """KPI: distribución de notas por asignatura, en rangos de `ancho_bin` puntos sobre 100.

    Devuelve (una fila por asignatura y rango): materia, rango (texto, p. ej. "80-90"),
    desde, hasta, estudiantes, porcentaje (de los estudiantes de la asignatura).
    """
    if matriculados:
        calificaciones = solo_matriculados(calificaciones, cruce)
    por_estudiante = nota_por_estudiante(calificaciones)

    bordes = list(range(0, 100 + ancho_bin, ancho_bin))
    rangos = pd.IntervalIndex.from_breaks(bordes, closed="left")
    # Un 100 exacto va al último rango
    nota = por_estudiante["nota_pct"].clip(upper=bordes[-1] - 1e-9)
    por_estudiante["rango"] = pd.cut(nota, bins=rangos)

    conteo = (
        por_estudiante.groupby(["materia", "rango"], observed=False).size()
        .rename("estudiantes").reset_index()
    )
    conteo["desde"] = conteo["rango"].map(lambda r: int(r.left))
    conteo["hasta"] = conteo["rango"].map(lambda r: int(r.right))
    conteo["rango"] = conteo["desde"].astype(str) + "-" + conteo["hasta"].astype(str)
    conteo["porcentaje"] = (conteo["estudiantes"] / conteo.groupby("materia")["estudiantes"].transform("sum") * 100).round(2)
    return conteo[["materia", "rango", "desde", "hasta", "estudiantes", "porcentaje"]]

In [ ]:
ANCHO_BIN = 10
kpi_distribucion = distribucion_de_notas(df_calificaciones, df_cruce, ANCHO_BIN)
kpi_distribucion.pivot(index="rango", columns="materia", values="estudiantes").loc[
    kpi_distribucion["rango"].unique()
]

### Imagen del histograma

Un panel por asignatura a partir de la misma tabla que consumirá Streamlit. La imagen se guarda en `data/kpis/histograma_notas.png`.

In [ ]:
import matplotlib.pyplot as plt

def graficar_histograma(distribucion: pd.DataFrame, ruta: Path | None = None):
    """Dibuja un histograma por asignatura a partir de `distribucion_de_notas`."""
    materias = distribucion["materia"].unique()
    fig, ejes = plt.subplots(1, len(materias), figsize=(5 * len(materias), 4), sharey=True, squeeze=False)
    for eje, materia in zip(ejes[0], materias):
        t = distribucion[distribucion["materia"] == materia]
        barras = eje.bar(t["rango"], t["estudiantes"], color="#1F5C6B", width=0.96)
        eje.bar_label(barras, padding=2, fontsize=8)
        eje.set_title(materia.capitalize())
        eje.set_xlabel("Nota (% del máximo posible)")
        eje.tick_params(axis="x", rotation=45)
        eje.spines[["top", "right"]].set_visible(False)
    ejes[0][0].set_ylabel("Estudiantes")
    fig.suptitle("Distribución de notas por asignatura", x=0.01, ha="left")
    fig.tight_layout()
    if ruta is not None:
        fig.savefig(ruta, dpi=150, bbox_inches="tight")
    return fig


fig = graficar_histograma(kpi_distribucion, DATA / "kpis" / "histograma_notas.png")
plt.show()

## 4. Porcentaje de estudiantes promocionados, regulares y libres

Cada asignatura define sus condiciones en puntos sobre un total distinto (Álgebra suma hasta 120; Probabilidad y Algoritmos, 100). Para poder compararlas y para no penalizar a quien todavía no rindió todo, los umbrales se expresan como **porcentaje del total posible** y se comparan con la nota del estudiante (`nota_pct`), que ya es un porcentaje de los puntos que tuvo oportunidad de obtener hasta ahora.

| Asignatura | Total | Regular desde | Promociona desde |
|---|---|---|---|
| Álgebra | 120 | 60 pts (50 %) | 85 pts (70,83 %) |
| Probabilidad y Estadística | 100 | 60 pts (60 %) | 85 pts (85 %) |
| Algoritmos y Estructuras de Datos | 100 | 60 pts (60 %) | 90 pts (90 %) |

Un estudiante con 95 % sobre lo rendido hasta ahora (por ejemplo 30 puntos de 30 posibles) queda **promocionado**, no libre. Es una clasificación según el desempeño actual, no una proyección del resultado final.

In [ ]:
# Condiciones de cada asignatura en puntos: (total posible, mínimo para regular, mínimo para promocionar)
CONDICIONES_PUNTOS = {
    "algebra": (120, 60, 85),
    "probabilidad": (100, 60, 85),
    "algoritmos": (100, 60, 90),
}
CONDICIONES = ["promocionado", "regular", "libre"]

# Mismos umbrales como porcentaje del total posible
UMBRALES_PCT = {
    materia: {"regular": regular / total * 100, "promocionado": promociona / total * 100}
    for materia, (total, regular, promociona) in CONDICIONES_PUNTOS.items()
}


def condicion_por_estudiante(calificaciones: pd.DataFrame, cruce: pd.DataFrame,
                             matriculados: bool = True) -> pd.DataFrame:
    """Condición (promocionado / regular / libre) de cada estudiante según su nota_pct.

    Devuelve: materia, usuario, instrumentos, puntos_obtenidos, puntos_posibles, nota_pct, condicion.
    """
    if matriculados:
        calificaciones = solo_matriculados(calificaciones, cruce)
    res = nota_por_estudiante(calificaciones)
    regular = res["materia"].map(lambda m: UMBRALES_PCT[m]["regular"])
    promociona = res["materia"].map(lambda m: UMBRALES_PCT[m]["promocionado"])
    # Pequeña tolerancia para que 85 puntos sobre 120 (70,8333...) no quede afuera por redondeo
    eps = 1e-9
    res["condicion"] = "libre"
    res.loc[res["nota_pct"] + eps >= regular, "condicion"] = "regular"
    res.loc[res["nota_pct"] + eps >= promociona, "condicion"] = "promocionado"
    return res


def condicion_academica(calificaciones: pd.DataFrame, cruce: pd.DataFrame,
                        matriculados: bool = True) -> pd.DataFrame:
    """KPI: estudiantes promocionados, regulares y libres por asignatura.

    Devuelve una fila por asignatura y condición (incluye las condiciones sin estudiantes):
    materia, condicion, estudiantes, porcentaje. Sirve directo para el gráfico de dona de Streamlit.
    """
    por_estudiante = condicion_por_estudiante(calificaciones, cruce, matriculados)
    conteo = (
        por_estudiante.groupby(["materia", "condicion"]).size().unstack(fill_value=0)
        .reindex(columns=CONDICIONES, fill_value=0)
        .stack().rename("estudiantes").reset_index()
    )
    conteo["porcentaje"] = (conteo["estudiantes"] / conteo.groupby("materia")["estudiantes"].transform("sum") * 100).round(2)
    return conteo

In [ ]:
kpi_condicion = condicion_academica(df_calificaciones, df_cruce)
kpi_condicion.pivot(index="materia", columns="condicion", values="porcentaje")[CONDICIONES]

**Control:** cantidades por condición y chequeo de los bordes (puntos exactos de cada umbral).

In [ ]:
display(kpi_condicion.pivot(index="materia", columns="condicion", values="estudiantes")[CONDICIONES])

# Bordes: nota justo en el umbral de regular y de promoción, y el caso parcial (30 de 30 puntos)
prueba = pd.DataFrame({
    "materia": ["algebra"] * 5 + ["algoritmos"] * 2,
    "usuario": ["a", "b", "c", "d", "e", "f", "g"],
    "nota_obtenida": [85, 84, 60, 59, 30, 90, 89],
    "nota_max": [120, 120, 120, 120, 30, 100, 100],
})
prueba_cruce = prueba[["materia", "usuario"]].assign(matriculado=True)
condicion_por_estudiante(prueba, prueba_cruce)[["materia", "usuario", "puntos_obtenidos", "puntos_posibles", "nota_pct", "condicion"]]

## 5. Comparación de resultados entre asignaturas

Este indicador no calcula una métrica nueva: reúne en una sola vista los KPI anteriores (promedio, distribución y condición académica) para poder contrastar las asignaturas, y agrega dos medidas que solo tienen sentido al comparar:

- **Dispersión** (`desvio_pct`): desvío estándar de la nota. Dos asignaturas con el mismo promedio pueden tener estudiantes muy parejos o muy dispares.
- **Diferencia contra el promedio general** (`dif_vs_general_pp`): puntos porcentuales respecto del promedio de todos los estudiantes de todas las asignaturas.
- **Avance de la evaluación** (`avance_evaluacion_pct`): qué parte del total de puntos está efectivamente evaluada (promedio por estudiante). Solo aplica a las asignaturas de puntaje acumulativo (hoy, Álgebra, con total 120); en Probabilidad y Algoritmos la nota es un promedio de instrumentos, no hay un total que completar y el valor queda vacío. Todo se expresa en porcentajes, así que los datos nuevos (un parcial, un recuperatorio) se suman solos, pero este valor indica qué tan completa es la comparación para Álgebra.

Al comparar hay que recordar que los umbrales para promocionar son distintos en cada asignatura (ver sección 4), por lo que "más promocionados" no equivale necesariamente a "mejor rendimiento".

Salidas pensadas para Streamlit:
- `comparar_asignaturas(...)`: tabla ancha, una fila por asignatura (para `st.dataframe` o tarjetas).
- `comparacion_en_formato_largo(...)`: columnas `materia`, `indicador`, `valor`, lista para el gráfico de barras agrupadas: `grouped_bar_chart({ind: g for ind, g in largo.groupby("indicador")}, "materia", "valor", ...)` de `ui/components/charts.py`.

In [ ]:
# Asignaturas cuya nota final es la suma de puntos (las demás promedian sus instrumentos)
ASIGNATURAS_ACUMULATIVAS = ["algebra"]


def comparar_asignaturas(calificaciones: pd.DataFrame, cruce: pd.DataFrame,
                         matriculados: bool = True) -> pd.DataFrame:
    """KPI: tabla comparativa de resultados por asignatura (todo en % del total posible).

    Devuelve una fila por asignatura: materia, estudiantes, promedio_pct, mediana_pct,
    desvio_pct, dif_vs_general_pp, pct_promocionados, pct_regulares, pct_libres,
    avance_evaluacion_pct.
    """
    por_estudiante = condicion_por_estudiante(calificaciones, cruce, matriculados)
    general = por_estudiante["nota_pct"].mean()

    res = por_estudiante.groupby("materia").agg(
        estudiantes=("usuario", "size"),
        promedio_pct=("nota_pct", "mean"),
        mediana_pct=("nota_pct", "median"),
        desvio_pct=("nota_pct", "std"),
        puntos_evaluados=("puntos_posibles", "mean"),
    )
    res["dif_vs_general_pp"] = res["promedio_pct"] - general
    # Solo tiene sentido donde la nota es una suma de puntos hasta un total (no en las que promedian)
    totales = pd.Series({m: CONDICIONES_PUNTOS[m][0] for m in ASIGNATURAS_ACUMULATIVAS})
    res["avance_evaluacion_pct"] = res["puntos_evaluados"] / totales * 100

    condicion = condicion_academica(calificaciones, cruce, matriculados).pivot(
        index="materia", columns="condicion", values="porcentaje"
    )
    res["pct_promocionados"] = condicion["promocionado"]
    res["pct_regulares"] = condicion["regular"]
    res["pct_libres"] = condicion["libre"]

    columnas = ["estudiantes", "promedio_pct", "mediana_pct", "desvio_pct", "dif_vs_general_pp",
                "pct_promocionados", "pct_regulares", "pct_libres", "avance_evaluacion_pct"]
    return res[columnas].round(2).reset_index()


def comparacion_en_formato_largo(comparacion: pd.DataFrame,
                                 indicadores: list[str] | None = None) -> pd.DataFrame:
    """Pasa la tabla comparativa a formato largo (materia, indicador, valor) para graficar."""
    indicadores = indicadores or ["promedio_pct", "pct_promocionados", "pct_regulares", "pct_libres"]
    return comparacion.melt(id_vars="materia", value_vars=indicadores, var_name="indicador", value_name="valor")

In [ ]:
kpi_comparacion = comparar_asignaturas(df_calificaciones, df_cruce)
kpi_comparacion

In [ ]:
kpi_comparacion_largo = comparacion_en_formato_largo(kpi_comparacion)
kpi_comparacion_largo